# Group 3: feature processing notebook

Этот ноутбук предназначен для исследовательской обработки признаков группы `3`.

Логика ячеек специально повторяет структуру `src/features/group_3/feature_processor.py`,
чтобы позже перенос был почти механическим.

In [1]:
from pathlib import Path
import pandas as pd
import sys

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

# теперь обычный импорт
from src.utils.spec_converter import create_feature_spec_template
from src.utils.io import load_feature_names_from_txt

In [3]:
# Пути относительно папки notebooks/group_3/
DATA_PATH = Path("../../data/raw/MIPT_hackathon_dataset.csv")
FEATURES_PATH = Path("../../data/feature_groups/features_group_3.txt")
OUTPUT_DIR = Path("../../notebook_outputs/group_3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
df = pd.read_csv(DATA_PATH)
feature_names = load_feature_names_from_txt(FEATURES_PATH)
block_df = df[feature_names].copy()

print("Block shape:", block_df.shape)
display(block_df.head())

Block shape: (5383, 19)


,lead_Условный отказ,lead_Поиск товаров GoSklad,lead_Метод доставки,current_status_id,lead_Оплачено клиентом,lead_Высота,days_handed_to_issued_pvz,lead_Стоимость доставки,lead_BANNER-SIZES,lead_Длина,lead_price,buyout_flag,contact_Город,lead_pipeline_id,lead_Дата возврата посылки на склад,lead_ПВЗ СДЭК,lead_type,lead_utm_group,lead_utm_referrer
0,NaN,NaN,NaN,142,NaN,4.0,3.03,NaN,NaN,30.0,5257,True,Санкт-Петербург,6892026,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,142,NaN,10.0,5.17,NaN,NaN,40.0,15780,True,Евпатория,6892026,NaN,NaN,NaN,dem,https://artraid-dem.ru/otek_clip4?utm_source=p...
2,NaN,NaN,NaN,142,NaN,11.0,4.98,NaN,NaN,43.0,14204,True,Барнаул,6892026,NaN,NaN,NaN,NaN,NaN
3,Посылка вернулась,NaN,NaN,143,NaN,6.0,3.28,NaN,NaN,40.0,8060,False,"Донецк, Ростовская область",6892026,1.764623e+09,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,142,NaN,6.0,3.81,NaN,NaN,40.0,8495,True,"Сургут, городской округ Сургут, Ханты-Мансийск...",6892026,NaN,NaN,NaN,dem,https://artraid-dem.ru/sustav_clip4?utm_source...


## Функции обработки признаков

Названия функций совпадают с private-методами из `feature_processor.py`.

In [ ]:
def _add_width_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Ширина" not in block_df.columns:
        return
    series = pd.to_numeric(block_df["lead_Ширина"], errors="coerce")
    result["lead_Ширина"] = series.fillna(series.median())


def _add_linear_height_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Линейная высота (см)" not in block_df.columns:
        return
    series = pd.to_numeric(block_df["lead_Линейная высота (см)"], errors="coerce")
    result["lead_Линейная высота (см)"] = series.fillna(series.median())
    result["lead_Линейная высота (см)__was_missing"] = series.isna().astype(int)


def _add_payment_type_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Вид оплаты" not in block_df.columns:
        return
    series = (
        block_df["lead_Вид оплаты"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
        .str.lower()
    )
    result["lead_Вид оплаты"] = series.replace({"": "unknown"}).astype(str)


def _add_returned_ts_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "returned_ts" not in block_df.columns:
        return
    ts = pd.to_datetime(block_df["returned_ts"], errors="coerce")
    result["returned_ts"] = ts.astype("string")
    result["returned_ts__is_present"] = ts.notna().astype(int)


def _add_delivery_service_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Служба доставки" not in block_df.columns:
        return
    series = (
        block_df["lead_Служба доставки"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
    )
    value_counts = series.value_counts(dropna=False)
    rare_values = value_counts[value_counts < 10].index
    result["lead_Служба доставки"] = series.replace(list(rare_values), "OTHER").astype(str)


def _add_default_feature(block_df: pd.DataFrame, result: pd.DataFrame, column: str) -> None:
    series = block_df[column]
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        result[column] = series.fillna("").astype(str)
    else:
        result[column] = series

In [ ]:
X_block = pd.DataFrame(index=block_df.index)

for column in block_df.columns:
    if column == "lead_Ширина":
        _add_width_feature(block_df, X_block)
    elif column == "lead_Линейная высота (см)":
        _add_linear_height_feature(block_df, X_block)
    elif column == "lead_Вид оплаты":
        _add_payment_type_feature(block_df, X_block)
    elif column == "returned_ts":
        _add_returned_ts_feature(block_df, X_block)
    elif column == "lead_Служба доставки":
        _add_delivery_service_feature(block_df, X_block)
    else:
        _add_default_feature(block_df, X_block, column)

print("Processed shape:", X_block.shape)
display(X_block.head())

In [ ]:
X_block.to_csv(OUTPUT_DIR / "X_block.csv", index=False)
feature_spec = create_feature_spec_template(X_block)
feature_spec.to_csv(OUTPUT_DIR / "feature_spec.csv", index=False)

print("Saved:")
print(OUTPUT_DIR / "X_block.csv")
print(OUTPUT_DIR / "feature_spec.csv")